# 00 · Data preparation and verification

Verifies the bundled IndoNLU SmSA and EmoT splits (SHA-256), reports split sizes and class distributions (the numbers used in Table II), and checks exact-duplicate overlap between splits. CPU is enough.

In [1]:
# ==== Setup (run first) ====
# SOURCE = "drive"  : the repository folder is on Google Drive (use this now)
# SOURCE = "github" : clone from GitHub (after the repository is public)
SOURCE = "drive"
DRIVE_REPO = "/content/drive/MyDrive/2026/Publikasi Sinta 2 Baru/mad-indo-sentiment"   # folder from the unzipped package
REPO_URL = "https://github.com/deepittalk-repo/mad-indo-sentiment.git"   # used only when SOURCE = "github"

import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")          # results persist in MyDrive/mad-indo-results
    if SOURCE == "drive":
        REPO_DIR = DRIVE_REPO
        assert os.path.exists(REPO_DIR), f"Folder not found: {REPO_DIR}. Upload the unzipped folder to MyDrive first."
    else:
        REPO_DIR = "/content/mad-indo-sentiment"
        if not os.path.exists(REPO_DIR):
            !git clone -q $REPO_URL $REPO_DIR
    %cd $REPO_DIR
    !pip -q install -r requirements.txt
else:
    # Local run: start Jupyter from the repository root or the notebooks/ folder.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
sys.path.insert(0, "src")

from mad_indo import data
cfg = data.load_config()
RESULTS = data.results_dir()
print("Results folder:", RESULTS)
LIMIT = cfg["limit_test_samples"]   # null in config = full test set


Mounted at /content/drive
/content/drive/MyDrive/2026/Publikasi Sinta 2 Baru/mad-indo-sentiment
Results folder: /content/drive/MyDrive/mad-indo-results


In [2]:
import pandas as pd
print(data.verify_checksums())
rows = []
for task in ["smsa", "emot"]:
    for split in ["train", "valid", "test"]:
        df = data.load_split(task, split)
        rows.append({"task": task, "split": split, "n": len(df), **df.label.value_counts().to_dict()})
table2 = pd.DataFrame(rows).fillna(0)
table2.to_csv(RESULTS / "table2_dataset_splits.csv", index=False)
table2

                                  file    ok
0              processed/emot_test.csv  True
1             processed/emot_train.csv  True
2             processed/emot_valid.csv  True
3              processed/smsa_test.csv  True
4             processed/smsa_train.csv  True
5             processed/smsa_valid.csv  True
6    raw/emot/emot_test_preprocess.csv  True
7   raw/emot/emot_train_preprocess.csv  True
8   raw/emot/emot_valid_preprocess.csv  True
9    raw/smsa/smsa_test_preprocess.tsv  True
10  raw/smsa/smsa_train_preprocess.tsv  True
11  raw/smsa/smsa_valid_preprocess.tsv  True


,task,split,n,positive,negative,neutral,anger,happy,sadness,fear,love
0,smsa,train,11000,6416.0,3436.0,1148.0,0.0,0.0,0.0,0.0,0.0
1,smsa,valid,1260,735.0,394.0,131.0,0.0,0.0,0.0,0.0,0.0
2,smsa,test,500,208.0,204.0,88.0,0.0,0.0,0.0,0.0,0.0
3,emot,train,3521,0.0,0.0,0.0,881.0,814.0,798.0,519.0,509.0
4,emot,valid,440,0.0,0.0,0.0,110.0,102.0,99.0,65.0,64.0
5,emot,test,440,0.0,0.0,0.0,110.0,101.0,100.0,65.0,64.0


In [3]:
# Exact-duplicate overlap with the training split (lower-cased, stripped)
for task in ["smsa", "emot"]:
    tr = set(data.load_split(task, "train").text.str.strip().str.lower())
    for split in ["valid", "test"]:
        df = data.load_split(task, split)
        print(task, split, "overlap with train:", df.text.str.strip().str.lower().isin(tr).sum())

smsa valid overlap with train: 14
smsa test overlap with train: 0
emot valid overlap with train: 3
emot test overlap with train: 1
